# 02 - Modèles de référence

Objectif : fixer le score à battre pour le LLM (Qwen3-4B-Instruct), avec deux modèles classiques qui prédisent l'Elo
à partir des variables d'entrée des prompts, pays inclus : une régression linéaire et un gradient boosting.

**Protocole.**
- Les données viennent de `data/prompts/` : mêmes parties, mêmes découpages et mêmes valeurs que les prompts. `first_moves` n'est pas utilisé : le LLM voit donc les 10 premiers coups en plus, alors que ces modèles n'en ont que le nom de l'ouverture.
- Les modèles sont entraînés sur train. Val sert uniquement à choisir les hyperparamètres du gradient boosting. Test n'est utilisé que pour l'évaluation.
- Une prédiction sous 100 est ramenée à 100, le plancher de l'Elo chess.com. Il faudra appliquer la même règle au LLM pour que la comparaison tienne.

**Lecture.** Le test compte 437 parties de 89 joueurs, et un joueur apporte jusqu'à 5 parties. Les intervalles de confiance sont donc calculés par joueur, comme dans `01_eda.ipynb`.

In [1]:
import json
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROMPTS_DIR = Path("data/prompts")
GAME_IDS_FILE = Path("data/game_ids.csv")  # correspondance locale, non versionnée : elle contient les pseudos

ELO_FLOOR = 100
BINS = [100, 400, 700, 1000, 1300, 1600, 1900, np.inf]
BIN_LABELS = ["100-400", "400-700", "700-1000", "1000-1300", "1300-1600", "1600-1900", "1900+"]
COUNTRY_ORDER = ["BR", "DE", "FR", "IN", "US"]

CATEGORICAL = ["country", "color", "opening_name", "eco", "result", "termination"]
NUMERIC = ["nb_moves", "nb_blunders", "nb_mistakes", "nb_inaccuracies", "acpl", "blunders_per_move"]
MIN_CATEGORY_GAMES = 10  # les modalités plus rares (1 827 ouvertures au total) sont regroupées


def load_split(name):
    with open(PROMPTS_DIR / f"{name}.jsonl", encoding="utf-8") as file:
        return pd.DataFrame([json.loads(line) for line in file])


train, val, test = load_split("train"), load_split("val"), load_split("test")
print(f"train : {len(train)}, val : {len(val)}, test : {len(test)} parties")
display(test.groupby("country").agg(parties=("elo", "size"), elo_moyen=("elo", "mean")).round(0))

train : 3332, val : 431, test : 437 parties


,parties,elo_moyen
country,,
BR,70,1200.0
DE,65,1236.0
FR,171,1164.0
IN,70,1148.0
US,61,1096.0


## 1. Entraînement

Les deux modèles reçoivent le même encodage. Le gradient boosting (`HistGradientBoostingRegressor`) a ses hyperparamètres
choisis sur val, avec l'erreur absolue moyenne comme critère : la référence doit être un modèle réglé, sans quoi la barre serait trop basse pour le LLM.

In [2]:
def preprocessor():
    encoder = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=MIN_CATEGORY_GAMES, sparse_output=False)
    return ColumnTransformer([("cat", encoder, CATEGORICAL), ("num", StandardScaler(), NUMERIC)])


def fit(model):
    return make_pipeline(preprocessor(), model).fit(train[CATEGORICAL + NUMERIC], train["elo"])


def predict(pipeline, df):
    return np.clip(pipeline.predict(df[CATEGORICAL + NUMERIC]), ELO_FLOOR, None)


grid = [{"learning_rate": lr, "max_depth": depth, "max_iter": n, "loss": loss}
        for lr, depth, n, loss in product([0.03, 0.1], [3, 6], [200, 500], ["squared_error", "absolute_error"])]
search = []
for params in grid:
    pipeline = fit(HistGradientBoostingRegressor(random_state=42, **params))
    search.append({**params, "mae_val": mean_absolute_error(val["elo"], predict(pipeline, val))})
search = pd.DataFrame(search).sort_values("mae_val")
display(search.head(5).round({"mae_val": 1}))

best = search.iloc[0][["learning_rate", "max_depth", "max_iter", "loss"]].to_dict()
best["max_depth"], best["max_iter"] = int(best["max_depth"]), int(best["max_iter"])

models = {
    "Constante": fit(DummyRegressor(strategy="median")),
    "Régression linéaire": fit(LinearRegression()),
    "Gradient boosting": fit(HistGradientBoostingRegressor(random_state=42, **best)),
}
nb_features = models["Régression linéaire"][0].get_feature_names_out().size
print(f"Hyperparamètres retenus : {best}")
print(f"{nb_features} variables après encodage")

,learning_rate,max_depth,max_iter,loss,mae_val
11,0.10,3,500,absolute_error,382.0
6,0.03,6,500,squared_error,382.9
10,0.10,3,500,squared_error,383.1
7,0.03,6,500,absolute_error,383.2
8,0.10,3,200,squared_error,384.1


Hyperparamètres retenus : {'learning_rate': 0.1, 'max_depth': 3, 'max_iter': 500, 'loss': 'absolute_error'}
131 variables après encodage


## 2. Résultats sur le test

`Constante` prédit toujours l'Elo médian du train : c'est le score d'un modèle qui n'a rien appris. Les quotas rendent les tranches à peu près équiprobables, donc une part de bonnes tranches autour de 1/7 (14 %) correspond au hasard.

In [3]:
def elo_bin(values):
    return pd.cut(values, BINS, right=False, labels=BIN_LABELS).astype(str)


def scores(df, prediction):
    return {
        "MAE": mean_absolute_error(df["elo"], prediction),
        "R²": r2_score(df["elo"], prediction),
        "Bonne tranche (%)": 100 * np.mean(elo_bin(prediction) == df["elo_bin"].to_numpy()),
    }


def score_table(df):
    """Scores par modèle, sur toutes les parties de df puis pays par pays."""
    rows = []
    for name, pipeline in models.items():
        prediction = predict(pipeline, df)
        rows.append({"Modèle": name, "Groupe": "Global", "Parties": len(df), **scores(df, prediction)})
        for country in COUNTRY_ORDER:
            mask = (df["country"] == country).to_numpy()
            rows.append({"Modèle": name, "Groupe": country, "Parties": int(mask.sum()), **scores(df[mask], prediction[mask])})
    table = pd.DataFrame(rows).set_index(["Modèle", "Groupe"])
    return table.reindex(list(models), level="Modèle")


results = score_table(test)
display(results.round({"MAE": 0, "R²": 3, "Bonne tranche (%)": 1}))

Parties    MAE     R²  Bonne tranche (%)
Modèle              Groupe                                          
Constante           Global      437  530.0 -0.002               15.3
                    BR           70  557.0 -0.008               18.6
                    DE           65  450.0 -0.034               16.9
                    FR          171  542.0 -0.001               16.4
                    IN           70  551.0 -0.000               14.3
                    US           61  526.0 -0.006                8.2
Régression linéaire Global      437  377.0  0.422               27.7
                    BR           70  434.0  0.384               27.1
                    DE           65  335.0  0.386               30.8
                    FR          171  386.0  0.440               27.5
                    IN           70  347.0  0.491               31.4
                    US           61  368.0  0.339               21.3
Gradient boosting   Global      437  369.0  0.440               26.1
                    BR           70  429.0  0.426               18.6
                    DE           65  333.0  0.394               32.3
                    FR          171  384.0  0.438               25.7
                    IN           70  328.0  0.505               31.4
                    US           61  346.0  0.405               23.0

### Sans les parties de leaderboard

Les conclusions de `01_eda.ipynb` demandent de recalculer les performances sans ces parties : elles pèsent plus lourd dans la tranche 1900+ en Inde et aux États-Unis qu'en France, ce qui peut déformer la comparaison entre pays.

In [4]:
results_no_lb = score_table(test[test["source"] != "leaderboard"])
display(results_no_lb.round({"MAE": 0, "R²": 3, "Bonne tranche (%)": 1}))

Parties    MAE     R²  Bonne tranche (%)
Modèle              Groupe                                          
Constante           Global      427  503.0 -0.000               15.7
                    BR           65  476.0 -0.011               20.0
                    DE           65  450.0 -0.034               16.9
                    FR          166  507.0 -0.002               16.9
                    IN           70  551.0 -0.000               14.3
                    US           61  526.0 -0.006                8.2
Régression linéaire Global      427  359.0  0.410               28.1
                    BR           65  380.0  0.309               29.2
                    DE           65  335.0  0.386               30.8
                    FR          166  361.0  0.429               27.7
                    IN           70  347.0  0.491               31.4
                    US           61  368.0  0.339               21.3
Gradient boosting   Global      427  355.0  0.406               26.0
                    BR           65  387.0  0.315               16.9
                    DE           65  333.0  0.394               32.3
                    FR          166  367.0  0.382               25.9
                    IN           70  328.0  0.505               31.4
                    US           61  346.0  0.405               23.0

### Par tranche d'Elo

Ce tableau montre où se concentre l'erreur : un modèle peut avoir une MAE moyenne correcte tout en ramenant les niveaux extrêmes vers le centre.

In [5]:
rows = []
for name in ["Régression linéaire", "Gradient boosting"]:
    prediction = predict(models[name], test)
    for label in BIN_LABELS:
        mask = (test["elo_bin"] == label).to_numpy()
        rows.append({"Modèle": name, "Tranche": label, "Parties": int(mask.sum()),
                     "MAE": mean_absolute_error(test["elo"][mask], prediction[mask]),
                     "Erreur moyenne (prédit - réel)": (prediction[mask] - test["elo"][mask]).mean(),
                     "Bonne tranche (%)": 100 * np.mean(elo_bin(prediction[mask]) == test["elo_bin"][mask].to_numpy())})
display(pd.DataFrame(rows).set_index(["Modèle", "Tranche"]).round(1))

Parties    MAE  Erreur moyenne (prédit - réel)  \
Modèle              Tranche                                                     
Régression linéaire 100-400         58  470.1                           430.2   
                    400-700         64  438.2                           399.5   
                    700-1000        62  303.5                           240.8   
                    1000-1300       67  247.3                           -33.1   
                    1300-1600       58  170.5                           -95.5   
                    1600-1900       68  375.4                          -347.5   
                    1900+           60  645.2                          -638.1   
Gradient boosting   100-400         58  418.4                           406.2   
                    400-700         64  393.4                           361.8   
                    700-1000        62  312.7                           220.0   
                    1000-1300       67  300.9                           -16.5   
                    1300-1600       58  199.2                          -103.6   
                    1600-1900       68  353.9                          -277.1   
                    1900+           60  613.6                          -599.8   

                               Bonne tranche (%)  
Modèle              Tranche                       
Régression linéaire 100-400                 24.1  
                    400-700                 14.1  
                    700-1000                22.6  
                    1000-1300               40.3  
                    1300-1600               65.5  
                    1600-1900               22.1  
                    1900+                    6.7  
Gradient boosting   100-400                 17.2  
                    400-700                 29.7  
                    700-1000                29.0  
                    1000-1300               29.9  
                    1300-1600               43.1  
                    1600-1900               23.5  
                    1900+                   10.0

## 3. Écarts entre pays : du signal ou du bruit ?

Intervalles de confiance à 95 % de la MAE, par bootstrap sur les joueurs : on retire des joueurs entiers, car leurs parties ne sont pas indépendantes. Cette cellule a besoin de `data/game_ids.csv`, produit par `build_prompts.py` et non versionné : sur un clone sans les données, elle est ignorée.

In [6]:
def bootstrap_mae(errors, players, n_draws=5000, seed=42):
    """Intervalle à 95 % de la MAE, en tirant les joueurs avec remise."""
    per_player = pd.DataFrame({"error": errors, "player": players}).groupby("player")["error"].agg(["sum", "size"])
    draws = np.random.default_rng(seed).integers(0, len(per_player), (n_draws, len(per_player)))
    maes = per_player["sum"].to_numpy()[draws].sum(axis=1) / per_player["size"].to_numpy()[draws].sum(axis=1)
    return np.percentile(maes, [2.5, 97.5])


if GAME_IDS_FILE.exists():
    players = test["game_id"].map(pd.read_csv(GAME_IDS_FILE, keep_default_na=False).set_index("game_id")["username"].str.lower())
    rows = []
    for name in ["Régression linéaire", "Gradient boosting"]:
        errors = np.abs(predict(models[name], test) - test["elo"].to_numpy())
        for group in ["Global"] + COUNTRY_ORDER:
            mask = np.ones(len(test), bool) if group == "Global" else (test["country"] == group).to_numpy()
            low, high = bootstrap_mae(errors[mask], players[mask].to_numpy())
            rows.append({"Modèle": name, "Groupe": group, "Joueurs": players[mask].nunique(),
                         "MAE": errors[mask].mean(), "IC 95 % bas": low, "IC 95 % haut": high})
    display(pd.DataFrame(rows).set_index(["Modèle", "Groupe"]).round(0))
else:
    print(f"{GAME_IDS_FILE} absent : lancer build_prompts.py pour obtenir les intervalles de confiance.")

Joueurs    MAE  IC 95 % bas  IC 95 % haut
Modèle              Groupe                                           
Régression linéaire Global       89  377.0        338.0         421.0
                    BR           14  434.0        303.0         585.0
                    DE           13  335.0        249.0         427.0
                    FR           35  386.0        330.0         452.0
                    IN           14  347.0        261.0         457.0
                    US           13  368.0        278.0         453.0
Gradient boosting   Global       89  369.0        336.0         406.0
                    BR           14  429.0        335.0         539.0
                    DE           13  333.0        263.0         406.0
                    FR           35  384.0        336.0         438.0
                    IN           14  328.0        242.0         442.0
                    US           13  346.0        272.0         422.0

## Conclusions

**1. Le score à battre pour le LLM, sur les 437 parties du test :**

| Modèle | MAE | R² | Bonne tranche |
|---|---|---|---|
| Constante (médiane du train) | 530 | 0,00 | 15,3 % |
| Régression linéaire | 377 | 0,42 | **27,7 %** |
| Gradient boosting | **369** | **0,44** | 26,1 % |

Sans les parties de leaderboard (427 parties), la MAE est de 359 pour la régression linéaire et de 355 pour le gradient boosting. Les deux modèles sont très proches : le gradient boosting gagne de 8 points de MAE, avec un intervalle global de 336 à 406, et perd sur la part de bonnes tranches. Pour dire que le LLM bat la référence, il faut donc qu'il dépasse le meilleur score sur chaque indicateur : MAE < 369, R² > 0,44 et plus de 27,7 % de bonnes tranches.

**2. Les modèles ramènent les prédictions vers le centre.** Ils surestiment fortement les faibles niveaux (environ +400 points en 100-400) et sous-estiment fortement les plus forts (environ -600 points en 1900+). La moyenne globale cache donc des erreurs très inégales selon le niveau : la MAE va de 170-200 en 1300-1600 à plus de 600 en 1900+. Il faudra regarder ce même tableau pour le LLM.

**3. Les écarts entre pays ne permettent pas de conclure.**
- Le gradient boosting va d'une MAE de 328 (Inde) à 429 (Brésil), mais chaque pays hors France ne compte que 13 ou 14 joueurs au test et les intervalles se chevauchent largement.
- Le score du Brésil dépend beaucoup de 5 parties de leaderboard : sans elles, sa MAE passe de 429 à 387.
- La part de bonnes tranches varie aussi selon le pays (18,6 % à 32,3 % pour le gradient boosting). Sur 60 à 70 parties, ces chiffres sont à lire avec la même prudence.

**4. Ces tableaux ne mesurent pas l'effet du pays.** Ils comparent des joueurs différents. Pour savoir si la prédiction change à cause du pays, il faut comparer les mêmes parties : c'est le rôle de `bias_test.jsonl` et de `prompt_no_country`.
